# Aircraft XML to Excel Preprocessing

This notebook reads aircraft data from an XML file, extracts relevant fields, computes the tire contact area, and exports the results to an Excel file.

In [8]:
import xml.etree.ElementTree as ET
from openpyxl.styles import Font as XLFont, Alignment
from openpyxl.utils import get_column_letter
import pandas as pd
import numpy as np
from pathlib import Path
import re

xml_path = Path("input_data/aircraft.xml")
tree = ET.parse(xml_path)
root = tree.getroot()

NS_F = "http://schemas.datacontract.org/2004/07/FaarFieldModel"
NS_A = "http://schemas.microsoft.com/2003/10/Serialization/Arrays"
NS_XSI = "http://www.w3.org/2001/XMLSchema-instance"

airplanes = root.find(f".//{{{NS_F}}}Airplanes")
assert airplanes is not None, "Could not find <Airplanes> in the XML."

def get_scalar(parent, tag, default=np.nan, cast=float):
    """Get text of a simple element <tag>value</tag> under parent."""
    el = parent.find(f"{{{NS_F}}}{tag}")
    if el is None:
        return default
    # handle xsi:nil="true"
    if el.attrib.get(f"{{{NS_XSI}}}nil", "").lower() == "true":
        return default
    if el.text is None:
        return default
    try:
        return cast(el.text.strip())
    except Exception:
        return default

def get_us(parent, tag, default=np.nan, cast=float):
    """Get the <us> child value under a unit-wrapped element <tag><si>..</si><us>..</us></tag>."""
    el = parent.find(f"{{{NS_F}}}{tag}")
    if el is None:
        return default
    us = el.find(f"{{{NS_F}}}us")
    if us is None or us.text is None:
        return default
    try:
        return cast(us.text.strip())
    except Exception:
        return default

In [9]:
# Tire rules herein are backed by the Bridgestone Aircraft Tire Application Tables and other manufacturer data.
# These are used to assign representative tire sizes/models based on airplane model

TIRE_RULES = [
    # BOEING
    # 737 Classics
    dict(pattern=r"\b(737-300|737-400|737-500)\b",
         size="H40×14.5-19",
         model="Bridgestone application table (main gear)",
         note="Doc: 737-300/400/500 main gear tire size H40×14.5-19",
         ribs=5),

    # 737-600/700 early entry (smaller than NG)
    dict(pattern=r"\b(737-6(?:00)?|737-700)\b",
         size="H43.5×16.0-21",
         model="Bridgestone application table (main gear)",
         note="Doc: 737-6/700 main gear tire size H43.5×16.0-21",
         ribs=5),

    # 737 NG family incl 900ER
    dict(pattern=r"\b(737-600|737-700|737-800|737-900|737-900ER|737NG)\b",
         size="H44.5×16.5-21",
         model="Bridgestone application table (main gear)",
         note="Doc: 737-600/700/800/900/900ER main gear tire size H44.5×16.5-21",
         ribs=5),

    # 737 MAX
    dict(pattern=r"\b(737\s*MAX-?(7|8|9|10)|MAX-?(7|8|9|10))\b",
         size="H44.5×16.5R21",
         model="Bridgestone application table (main gear)",
         note="Doc: 737 MAX main gear tire size H44.5×16.5R21",
         ribs=5),

    # 757
    dict(pattern=r"\b(757-200|757-300|B757)\b",
         size="H40×14.5-19",
         model="Bridgestone application table (main gear)",
         note="Doc: 757-200/300 main gear tire size H40×14.5-19",
         ribs=5),

    # 767 split
    dict(pattern=r"\b(767-200)\b",
         size="H45.5×17.0-20",
         model="Bridgestone application table (main gear)",
         note="Doc: 767-200 main gear tire size H45.5×17.0-20",
         ribs=5),

    dict(pattern=r"\b(767-200ER|767-300|767-300ER|767-400ER|B767)\b",
         size="H46×18.0-20",
         model="Bridgestone application table (main gear)",
         note="Doc: 767-200ER/300(/ER/HGW) main gear tire size H46×18.0-20; 767-400ER uses 50×20.0R22",
         ribs=5),

    # 747 family split
    dict(pattern=r"\b(747-400SR|747-400|B744)\b",
         size="H49.5×19.0-22",
         model="Bridgestone application table (main gear)",
         note="Doc: 747-400 main gear tire size H49.5×19.0-22",
         ribs=7),

    dict(pattern=r"\b(747-400ER)\b",
         size="50×20.0R22",
         model="Bridgestone application table (main gear)",
         note="Doc: 747-400ER main gear tire size 50×20.0R22",
         ribs=7),

    dict(pattern=r"\b(747-8)\b",
         size="52×21.0R22",
         model="Bridgestone application table (main gear)",
         note="Doc: 747-8 main gear tire size 52×21.0R22",
         ribs=7),

    # 777 split
    dict(pattern=r"\b(777F|777-200LR|777-300ER|77W|200LR)\b",
         size="52×21.0R22",
         model="Bridgestone application table (main gear)",
         note="Doc: 777F and 777-200LR/300ER main gear tire size 52×21.0R22",
         ribs=7),

    dict(pattern=r"\b(777-8|777-9)\b",
         size="52×21.0R22",
         model="Bridgestone application table (main gear)",
         note="Doc: 777-8/9 main gear tire size 52×21.0R22",
         ribs=7),

    dict(pattern=r"\b(B777|777)\b",
         size="50×20.0R22",
         model="Bridgestone application table (main gear)",
         note="Doc: base 777 listed with main gear tire size 50×20.0R22",
         ribs=7),

    # 787 split
    dict(pattern=r"\b(787-8)\b",
         size="50×20.0R22",
         model="Bridgestone application table (main gear)",
         note="Doc: 787-8 main gear tire size 50×20.0R22",
         ribs=7),

    dict(pattern=r"\b(787-9|787-10)\b",
         size="54×21.0R23",
         model="Bridgestone application table (main gear)",
         note="Doc: 787-9/10 main gear tire size 54×21.0R23",
         ribs=7),

    # AIRBUS
    dict(pattern=r"\b(A318|A319|A320|A320NEO|A319NEO)\b",
         size="46×17R20",
         model="Bridgestone application table (main gear)",
         note="Doc: A320/A319/A318 and A320neo/A319neo main gear includes 46×17R20",
         ribs=5),

    dict(pattern=r"\b(A321|A321NEO)\b",
         size="1270×455R22",
         model="Bridgestone application table (main gear)",
         note="Doc: A321/A321neo main gear includes 1270×455R22",
         ribs=5),

    dict(pattern=r"\b(A330NEO|A330)\b",
         size="1400×530R23",
         model="Bridgestone application table (main gear)",
         note="Doc: A330 and A330neo main gear includes 1400×530R23",
         ribs=7),

    dict(pattern=r"\b(A340-200|A340-300|A340-500|A340-600|A340)\b",
         size="1400×530R23",
         model="Bridgestone application table (main gear)",
         note="Doc: A340 variants main gear includes 1400×530R23",
         ribs=7),

    dict(pattern=r"\b(A350-800|A350-900|A350\s*XWB-?(800|900))\b",
         size="1400×530R23",
         model="Bridgestone application table (main gear)",
         note="Doc: A350-800/900 main gear includes 1400×530R23",
         ribs=7),

    dict(pattern=r"\b(A350-1000|A350\s*XWB-?1000)\b",
         size="50×20.0R22",
         model="Bridgestone application table (main gear)",
         note="Doc: A350-1000 main gear includes 50×20.0R22",
         ribs=7),

    dict(pattern=r"\b(A380|A380-800)\b",
         size="1400×530R23",
         model="Bridgestone application table (main gear)",
         note="Doc: A380-800 main gear includes 1400×530R23",
         ribs=7),
]


RIB_PROFILES = {
    7: {
        "Ribs (mm)": "50;35;40;90;40;35;50",
        "Load Factor": "0.24;0.08;0.08;0.20;0.08;0.08;0.24",
        "Stress Factor": "1.80;1.10;1.10;1.10;1.10;1.10;1.80",
    },
    5: {
        "Ribs (mm)": "70;45;90;45;70",
        "Load Factor": "0.28;0.11;0.22;0.11;0.28",
        "Stress Factor": "1.80;1.10;1.10;1.10;1.80",
    },
    3: {
        "Ribs (mm)": "70;90;70",
        "Load Factor": "0.35;0.30;0.35",
        "Stress Factor": "1.80;1.10;1.80",
    },
}

RIB_CLASSES = (3, 5, 7)
WIDTH_3_MAX = 9.0
WIDTH_7_MIN = 18.0
AREA_3_MAX  = 120.0
AREA_7_MIN  = 300.0


def _to_float_or_nan(x):
    try:
        v = float(x)
        if np.isnan(v) or v <= 0:
            return np.nan
        return v
    except Exception:
        return np.nan

def rib_fallback(area_in2, width_in):
    """
    Returns: (ribs_class, confidence, rationale)
    ribs_class is a PROFILE CLASS (3/5/7), not a literal rib count.
    """
    area  = _to_float_or_nan(area_in2)
    width = _to_float_or_nan(width_in)

    if pd.isna(area) and pd.isna(width):
        return 5, "LOW", "No area/width → default profile class = 5"

    # Vote from width
    width_vote = None
    if not pd.isna(width):
        if width < WIDTH_3_MAX:
            width_vote = 3
        elif width > WIDTH_7_MIN:
            width_vote = 7
        else:
            width_vote = 5

    # Vote from area
    area_vote = None
    if not pd.isna(area):
        if area < AREA_3_MAX:
            area_vote = 3
        elif area > AREA_7_MIN:
            area_vote = 7
        else:
            area_vote = 5

    # Combine votes
    votes = [v for v in [width_vote, area_vote] if v is not None]

    if len(votes) == 1:
        v = votes[0]
        src = "width" if width_vote is not None else "area"
        return v, "MED", f"Single-signal ({src}) → {v}"

    # If both present:
    if width_vote == area_vote:
        return width_vote, "HIGH", f"Width and area agree → {width_vote}"

    # Disagreement: choose conservative middle class and lower confidence
    # (or you can choose by whichever signal is 'closer' to thresholds)
    return 5, "LOW", f"Conflict: width→{width_vote}, area→{area_vote} → choose 5"

# ---------- RULE MATCHING HARDENING ----------

def norm_name(name: str) -> str:
    s = (name or "").upper()
    s = s.replace("–", "-").replace("−", "-")
    # normalize separators to spaces, keep alphanumerics and hyphen
    s = re.sub(r"[^A-Z0-9\-]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Precompile once (do this right after defining TIRE_RULES)
for rule in TIRE_RULES:
    rule["rx"] = re.compile(rule["pattern"], flags=re.IGNORECASE)

def assign_rep_tire_and_ribs(airplane_name, area_in2, width_in):
    s = norm_name(airplane_name)

    for rule in TIRE_RULES:
        if rule["rx"].search(s):
            # If you're treating ribs as "profile class", say so explicitly
            note = f"{rule['note']} | ribs={rule['ribs']} (profile class)"
            return rule["size"], rule["model"], note, rule["ribs"], "FamilyRule", "MED"

    ribs, conf, why = rib_fallback(area_in2, width_in)
    note = f"Heuristic ({conf}): {why} | thresholds: area<{AREA_3_MAX}→3, area>{AREA_7_MIN}→7; width<{WIDTH_3_MAX}→3, width>{WIDTH_7_MIN}→7"
    return "", "", note, ribs, "Heuristic", conf

In [10]:
rows = []
for ap in list(airplanes):
    # only keep the AirplaneInfo blocks
    if ap.tag != f"{{{NS_A}}}anyType":
        continue
    if ap.attrib.get(f"{{{NS_XSI}}}type") != "AirplaneInfo":
        continue

    name = ap.find(f"{{{NS_F}}}Name")
    name = name.text.strip() if (name is not None and name.text) else ""

    # Requested fields
    gw_lbs = get_us(ap, "_GrossWeight")          # "Gross Taxi Weight (lbs)"
    tire_area = get_us(ap, "TireArea", default=0.0)
    tire_len = get_us(ap, "TireLength", default=0.0)
    tire_wid = get_us(ap, "TireWidth", default=0.0)

    # TirePressureF is often nil; default to 0 per your requirement
    tire_pressure = get_us(ap, "Cp", default=0.0)
    mg_percent = get_scalar(ap, "MgPercent")
    mg_percent_pcn = get_scalar(ap, "MgPercentPCN")

    num_gear = get_scalar(ap, "NumberGear", cast=int)
    num_tracks = get_scalar(ap, "NumberTireTracks", cast=int, default=1.0)
    num_wheels = get_scalar(ap, "NumberWheels", cast=int)

    rows.append({
        "Airplane Name": name,
        "Gross Taxi Weight (lbs)": gw_lbs,
        "Tire Pressure (psi)": tire_pressure,
        "Percent GW on Gear": mg_percent,
        "MgPercentPCN": mg_percent_pcn,
        "Number Gear": num_gear,
        "Number Tire Tracks": num_tracks,
        "Number Wheels": num_wheels,
        "Tire Contact Width (in.)": tire_wid,
        "Tire Contact Length (in.)": tire_len,
        "Tire Contact Area (in.^2)": tire_area,
    })

df = pd.DataFrame(rows)

df["Number Tire Tracks"] = df["Number Tire Tracks"].replace(np.nan, 1) # Use 1 as default, to avoid division by 0 later

# Manual Entries
df = pd.concat([
    df,
    pd.DataFrame([{
        "Airplane Name": "ICT-B777-300ER",
        "Gross Taxi Weight (lbs)": 777000,
        "Tire Pressure (psi)": 200.0,
        "Percent GW on Gear": 0.525,
        "MgPercentPCN": 0.2660,
        "Number Gear": 3,
        "Number Tire Tracks": 4,
        "Number Wheels": 6,
        "Tire Contact Width (in.)": 13.39,
        "Tire Contact Length (in.)": 14.96,
        "Tire Contact Area (in.^2)": 200.0,
    }, {
        "Airplane Name": "ICT-A380-800",
        "Gross Taxi Weight (lbs)": 1239000,
        "Tire Pressure (psi)": 210.3,
        "Percent GW on Gear": 0.38,
        "MgPercentPCN": 0.19,
        "Number Gear": 3,
        "Number Tire Tracks": 4,
        "Number Wheels": 4,
        "Tire Contact Width (in.)": 14.17,
        "Tire Contact Length (in.)": 22.05,
        "Tire Contact Area (in.^2)": 270.00,
    }])
], ignore_index=True)


df["Load (lbs)"] = (df["Gross Taxi Weight (lbs)"] * df["MgPercentPCN"]) / df["Number Wheels"]
df["Load (N)"] = df["Load (lbs)"] * 4.44822
df["Tire Pressure (MPa)"] = df["Tire Pressure (psi)"] * 0.00689476
df["Tire Contact Length (mm)"] = df["Tire Contact Length (in.)"] * 25.4
df["Tire Contact Width (mm)"] = df["Tire Contact Width (in.)"] * 25.4
df["Tire Contact Area (cm.^2)"] = df["Tire Contact Area (in.^2)"] * 6.4516
# Optional: round the contact geometry to match your example output style
df["Tire Contact Width (in.)"] = df["Tire Contact Width (in.)"].round(1)
df["Tire Contact Length (in.)"] = df["Tire Contact Length (in.)"].round(1)
df["Tire Contact Area (in.^2)"] = df["Tire Contact Area (in.^2)"].round(1)


#=====
# Number of Ribs Assigned
#=====
rep = df.apply(
    lambda r: assign_rep_tire_and_ribs(
        r.get("Airplane Name", ""),
        r.get("Tire Contact Area (in.^2)", np.nan),
        r.get("Tire Contact Width (in.)", np.nan)
    ),
    axis=1,
    result_type="expand"
)

rep.columns = [
    "Repr. Tire Size",
    "Repr. Tire Model",
    "Repr. Model SourceNote",
    "Est_NRibs",
    "Rib_Method",
    "Rib_Confidence"
]

insert_at = df.columns.get_loc("Tire Contact Area (in.^2)") + 1
for col in rep.columns[::-1]:
    df.insert(insert_at, col, rep[col])
    
df["Est_NRibs"] = pd.to_numeric(df["Est_NRibs"], errors="coerce").fillna(5).astype(int)
#=====
#=====

# Categorization
df["Ribs (mm)"] = df["Est_NRibs"].map(lambda n: RIB_PROFILES.get(n, RIB_PROFILES[5])["Ribs (mm)"])
df["Load Factor"] = df["Est_NRibs"].map(lambda n: RIB_PROFILES.get(n, RIB_PROFILES[5])["Load Factor"])
df["Stress Factor"] = df["Est_NRibs"].map(lambda n: RIB_PROFILES.get(n, RIB_PROFILES[5])["Stress Factor"])

# Replace the value in Ribs(mm), Load Factor, and Stress Factor columns for this entry: ICT-A380-800
df.loc[df["Airplane Name"] == "ICT-A380-800", "Ribs (mm)"] = "70;45;90;45;70"
df.loc[df["Airplane Name"] == "ICT-A380-800", "Load Factor"] = "0.28;0.11;0.22;0.11;0.28"
df.loc[df["Airplane Name"] == "ICT-A380-800", "Stress Factor"] = "1.80;1.10;1.10;1.10;1.80"


num_formats = {
    "Gross Taxi Weight (lbs)": "0",
    "Tire Pressure (psi)": "0.0",
    "Percent GW on Gear": "0.0000",
    "MgPercentPCN": "0.0000",
    "Number Gear": "0",
    "Number Tire Tracks": "0",
    "Number Wheels": "0",
    "Tire Contact Width (in.)": "0.0",
    "Tire Contact Length (in.)": "0.0",
    "Tire Contact Area (in.^2)": "0.00",
    "Load (lbs)": "0",
    "Load (N)": "0",
    "Tire Pressure (MPa)": "0.000",
    "Tire Contact Length (mm)": "0",
    "Tire Contact Width (mm)": "0",
    "Tire Contact Area (cm.^2)": "0.00",
    "Est_NRibs": "0"
}

for c in num_formats:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")


out_path = Path("input_data/aircraft.xlsx")
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Aircraft", index=False)

    ws = writer.sheets["Aircraft"]

    base_font    = XLFont(name="Times New Roman", size=11)
    header_font  = XLFont(name="Times New Roman", size=11, bold=True)
    center_align = Alignment(horizontal="center", vertical="center")

    # Font + alignment
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row,
                            min_col=1, max_col=ws.max_column):
        for cell in row:
            cell.font = header_font if cell.row == 1 else base_font
            cell.alignment = center_align

    # Map headers -> column index
    header_to_col = {
        ws.cell(row=1, column=col).value: col
        for col in range(1, ws.max_column + 1)
    }

    # Apply numeric formats (and coerce stray strings to numbers)
    for col_name, fmt in num_formats.items():
        col_idx = header_to_col.get(col_name)
        if not col_idx:
            continue
        for r in range(2, ws.max_row + 1):
            cell = ws.cell(row=r, column=col_idx)
            if isinstance(cell.value, str):
                try:
                    cell.value = float(cell.value)
                except Exception:
                    pass
            cell.number_format = fmt

    # Auto-fit widths
    for col_idx in range(1, ws.max_column + 1):
        max_len = 0
        for r in range(1, ws.max_row + 1):
            v = ws.cell(row=r, column=col_idx).value
            if v is None:
                continue
            if isinstance(v, float) and pd.isna(v):
                continue
            max_len = max(max_len, len(str(v)))
        ws.column_dimensions[get_column_letter(col_idx)].width = max_len + 2

In [11]:
# Visualize the first rows of the dataframe
df.head()

,Airplane Name,Gross Taxi Weight (lbs),Tire Pressure (psi),Percent GW on Gear,MgPercentPCN,Number Gear,Number Tire Tracks,Number Wheels,Tire Contact Width (in.),Tire Contact Length (in.),...,Rib_Confidence,Load (lbs),Load (N),Tire Pressure (MPa),Tire Contact Length (mm),Tire Contact Width (mm),Tire Contact Area (cm.^2),Ribs (mm),Load Factor,Stress Factor
0,SWL-2,2000.0,30.0,1.0,1.0,1,1,1,7.3,11.7,...,HIGH,2000.0,8896.44,0.206843,296.007901,185.004938,430.106650,70;90;70,0.35;0.30;0.35,1.80;1.10;1.80
1,SWL-5,5000.0,45.0,1.0,1.0,1,1,1,9.4,15.0,...,LOW,5000.0,22241.10,0.310264,382.144581,238.840363,716.844466,70;45;90;45;70,0.28;0.11;0.22;0.11;0.28,1.80;1.10;1.10;1.10;1.80
2,SWL-10,10000.0,50.0,1.0,1.0,1,1,1,12.6,20.2,...,HIGH,10000.0,44482.20,0.344738,512.700760,320.437975,1290.320000,70;45;90;45;70,0.28;0.11;0.22;0.11;0.28,1.80;1.10;1.10;1.10;1.80
3,Single Wheel 2,2000.0,30.0,1.0,0.5,1,1,1,0.0,0.0,...,LOW,1000.0,4448.22,0.206843,0.000000,0.000000,0.000000,70;45;90;45;70,0.28;0.11;0.22;0.11;0.28,1.80;1.10;1.10;1.10;1.80
4,Single Wheel 5,5000.0,45.0,1.0,0.5,1,1,1,0.0,0.0,...,LOW,2500.0,11120.55,0.310264,0.000000,0.000000,0.000000,70;45;90;45;70,0.28;0.11;0.22;0.11;0.28,1.80;1.10;1.10;1.10;1.80
